# Data Leakage & Pipelines - Exercise

**Case: predicting customer churn at *StreamBox*, a video streaming subscription service.**

The marketing team wants a model that flags customers who are likely to cancel their subscription (**churn**) in the coming month, so they can contact them in time. They gave you an export of 2,000 customers from their CRM system.

In this exercise you will:
1. Explore the data and **spot a target-leakage feature**;
2. See how **preprocessing outside cross-validation** (imputation, scaling and especially *feature selection*) can make your results look better than they really are;
3. Build a **proper scikit-learn `Pipeline` with a `ColumnTransformer`**, evaluate it with cross-validation on the training set and once on the test set;
4. Tune the pipeline with **`GridSearchCV`**;
5. Think about some new **data leakage scenarios**.

The data is synthetic and generated inside the notebook, so everybody works with exactly the same dataset (no download needed).

---
---
---
## Complete the tasks in bold

## 0 - Setup

**TASK: Run the cell below to import the libraries you will need. Later on you may need to import more from scikit-learn.**

In [ ]:
import numpy as np # To do some calculations
import pandas as pd # To work with dataframes
import matplotlib.pyplot as plt # To make plots

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, GridSearchCV
from sklearn.linear_model import LogisticRegression # The machine learning model

## 1 - Data Generation and Exploration

The cell below generates the CRM export. Every row is one customer. The columns are:

| column | meaning |
|---|---|
| `customer_id` | unique customer number |
| `age` | age of the customer (years) |
| `tenure_months` | how many months the customer has been subscribed |
| `monthly_fee` | monthly subscription fee (EUR) |
| `support_calls` | number of calls to the helpdesk in the last 3 months |
| `weekly_usage_hours` | average hours of streaming per week |
| `contract_type` | `Monthly`, `Yearly` or `Two-year` |
| `payment_method` | `Credit card`, `Bank transfer`, `PayPal` or `Voucher` |
| `region` | `North`, `South`, `East` or `West` |
| `retention_offer_sent` | 1 if the retention team sent the customer a discount offer, else 0 |
| `churn` | **target**: 1 if the customer cancelled, else 0 |

**TASK: Run the cell below to generate the dataset `df`. Do not change the code, so that your numbers match the solution.**

In [ ]:
rng = np.random.default_rng(42)
n = 2000

age = np.clip(rng.normal(40, 12, n), 18, 80).round()
tenure = rng.integers(1, 73, n)
fee = np.clip(rng.normal(45, 15, n), 10, 120).round(2)
calls = rng.poisson(1.5, n)
usage = rng.gamma(2.0, 2.5, n).round(1)
contract = rng.choice(["Monthly", "Yearly", "Two-year"], n, p=[0.55, 0.30, 0.15])
payment = rng.choice(["Credit card", "Bank transfer", "PayPal", "Voucher"], n, p=[0.4, 0.3, 0.2, 0.1]).astype(object)
region = rng.choice(["North", "South", "East", "West"], n)

# the (hidden) true relationship between the features and churn
logit = (-1.0 + 0.03 * (fee - 45) - 0.04 * (tenure - 36) + 0.45 * (calls - 1.5) - 0.15 * (usage - 5)
         + np.select([contract == "Monthly", contract == "Yearly"], [1.0, -0.5], -1.5)
         + 0.6 * (payment == "Voucher"))
churn = rng.binomial(1, 1 / (1 + np.exp(-logit)))

# the retention team sends an offer when a customer calls to cancel...
offer = np.where(churn == 1, rng.binomial(1, 0.93, n), rng.binomial(1, 0.03, n))

# some values are missing in the CRM export
age[rng.random(n) < 0.08] = np.nan
usage[rng.random(n) < 0.10] = np.nan
payment[rng.random(n) < 0.05] = np.nan

df = pd.DataFrame({"customer_id": np.arange(10001, 10001 + n), "age": age, "tenure_months": tenure,
                   "monthly_fee": fee, "support_calls": calls, "weekly_usage_hours": usage,
                   "contract_type": contract, "payment_method": payment, "region": region,
                   "retention_offer_sent": offer, "churn": churn})

**TASK: Display the first rows of `df` and print its shape.** *(Expected: 2000 rows and 11 columns.)*

**TASK: Count the number of missing values per column.** *(Expected: `age` 153, `weekly_usage_hours` 185, `payment_method` 105.)*

**TASK: Calculate the churn rate (the proportion of customers with `churn == 1`) and show the value counts of the three categorical columns.** *(Expected churn rate: 0.3815.)*

**QUESTION: Summarise the data. Which columns are numeric and which are categorical? Which columns have missing values? Which column should never be used as a feature, regardless of leakage? Which preprocessing steps will be needed before a `LogisticRegression` can be trained?**

*Your answer:*

## 2 - Target Leakage

Remember from class: **target leakage** happens when a feature contains information that would *not* be available at the moment you make the prediction, typically because it is created *after* (or as a consequence of) the outcome. It often results in suspiciously good performance.

**TASK: Group `df` by `churn` and calculate the mean of every numeric column (drop `customer_id`). Which feature differs extremely between the two groups?**

**TASK: Make a crosstab (`pd.crosstab`) of `retention_offer_sent` against `churn`.**

We will use a stratified 5-fold cross-validation with shuffling for all experiments, so that every fold has roughly the same churn rate and the results are reproducible.

**TASK: Run the cell below to define the cross-validation strategy `cv` that we will use throughout the notebook.**

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

**TASK: Use `cross_val_score` with a `LogisticRegression`, `cv=cv` and `scoring="roc_auc"` to compute the mean cross-validated AUC of (a) a model with only `retention_offer_sent` and (b) a model with only `tenure_months`, `monthly_fee` and `support_calls`.** *(Expected: (a) ≈ 0.956, (b) ≈ 0.703.)*

### What happens in production?

When the marketing team uses the model at the start of the month, the retention team has **not yet** sent any offers: the offer is only sent *after* a customer calls to cancel. So at prediction time `retention_offer_sent` is 0 for everybody.

**TASK: Split `df[honest_features + ["retention_offer_sent"]]` and `y` into a train and test set (`test_size=0.2`, `random_state=42`, `stratify=y`). Train a `LogisticRegression` on the training set. Report the accuracy and the recall of the churners (`recall_score`) on the test set (1) as it is and (2) after setting `retention_offer_sent` to 0 in a copy of the test set.** *(Expected: accuracy ≈ 0.955 and recall ≈ 0.94 on the historical test set; in production accuracy ≈ 0.62 and recall 0.00.)*

**QUESTION: Why is `retention_offer_sent` a leaky feature? Explain (1) how you could detect it, (2) why it is not available at prediction time and (3) what the production simulation shows.**

*Your answer:*

**TASK: Create `X` by dropping `customer_id`, `retention_offer_sent` and `churn` from `df`. Also define the lists `numeric_features` and `categorical_features`.**

## 3 - Train-Test Split (Critical First Step!)

As in class: **before any preprocessing**, we put a test set aside. It will only be used once, at the very end, to get an unbiased estimate of the performance.

**TASK: Split `X` and `y` into `X_train`, `X_test`, `y_train`, `y_test` with `test_size=0.2`, `random_state=42` and `stratify=y`. Print the shapes and the churn rate in both sets.** *(Expected: 1600 and 400 rows, churn rate ≈ 0.38 in both.)*

## 4 - Preprocessing Leakage

Splitting the data is not enough: **all preprocessing steps must be learned from the training data only**, and inside cross-validation that means *from the training folds only*. If you fit an imputer, scaler or feature selector on all the data first and only then run cross-validation, information from the validation folds leaks into the training.

### 4.1 Imputation and scaling

**TASK (wrong way): Fit a `SimpleImputer(strategy="median")` and a `StandardScaler` on all numeric columns of `X_train`, transform the data and then compute the cross-validated AUC of a `LogisticRegression` on the transformed data.** *(Expected: ≈ 0.7348.)*

**TASK (right way): Build a pipeline (`make_pipeline` or `Pipeline`) with the imputer, the scaler and the `LogisticRegression`, and compute its cross-validated AUC on the numeric columns of `X_train`.** *(Expected: ≈ 0.7348.)*

**QUESTION: How big is the difference between the wrong and the right way here? Why do you think it is so small? Does that mean preprocessing outside the cross-validation is fine?**

*Your answer:*

### 4.2 Feature selection

The web analytics team offers you **2,000 extra "clickstream" features** for a pilot group of 150 customers. They hope that some of them predict churn. In reality (but you don't know that yet...) these features are **pure random noise**.

As in the class notebook, a model trained on random features should have an AUC of about **0.5**.

**TASK: Run the cell below to generate the clickstream data for the pilot group.**

In [ ]:
rng_clicks = np.random.default_rng(7)
X_clicks = pd.DataFrame(rng_clicks.normal(size=(150, 2000)),
                        columns=[f"click_{i:04d}" for i in range(2000)])
y_clicks = df["churn"].iloc[:150].reset_index(drop=True)
print(X_clicks.shape, f"churn rate pilot: {y_clicks.mean():.3f}")

**TASK (wrong way): Use `SelectKBest(k=20)` fitted on all of `X_clicks` and `y_clicks` to select 20 features. Then compute the cross-validated AUC (`cv=cv`) of a `LogisticRegression` on the selected features.** *(Expected: ≈ 0.88.)*

**TASK (right way): Put `SelectKBest(k=20)` and the `LogisticRegression` in one pipeline and compute its cross-validated AUC on all 2000 clickstream features.** *(Expected: ≈ 0.42, i.e. no better than random.)*

**TASK: Make a bar chart comparing the mean AUC of the wrong and right way of feature selection. Add a dashed horizontal line at 0.5 (random classifier).**

**QUESTION: Compare the two AUCs. Why does feature selection outside the cross-validation give such an optimistic result on random data, while the effect of imputation/scaling in 4.1 was negligible? What would happen if the marketing team bought these clickstream features based on the "wrong" result?**

*Your answer:*

## 5 - A Proper Pipeline with `ColumnTransformer`

Now we build the full preprocessing + model pipeline for the churn data, as in the class notebook *6 - Pipelines*:
- **numeric features**: impute with the median, then standardise;
- **categorical features**: impute with the most frequent value, then one-hot encode (`handle_unknown="ignore"`);
- **model**: `LogisticRegression(max_iter=1000)`.

**TASK: Create a `numeric_transformer` (Pipeline with `SimpleImputer(strategy="median")` named `"imputer"` and `StandardScaler` named `"scaler"`) and a `categorical_transformer` (Pipeline with `SimpleImputer(strategy="most_frequent")` named `"imputer"` and `OneHotEncoder(handle_unknown="ignore")` named `"encoder"`).**

**TASK: Combine both transformers in a `ColumnTransformer` called `preprocessor` (names `"num"` and `"cat"`), and join it with the classifier in a `Pipeline` called `model` (steps `"preprocessor"` and `"classifier"`). Display the pipeline.**

**TASK: Compute the 5-fold cross-validated AUC of `model` on `X_train`, `y_train` (`cv=cv`, `scoring="roc_auc"`). Print the mean and standard deviation.** *(Expected: ≈ 0.809 ± 0.036.)*

**TASK: Fit `model` on the training set. Compute the AUC (`roc_auc_score`, using `predict_proba`) and the accuracy on the test set, and plot the ROC curve on the test set (e.g. with `RocCurveDisplay.from_estimator`).** *(Expected: test AUC ≈ 0.761, accuracy ≈ 0.71.)*

**TASK: Show the feature names created by the fitted preprocessor (`get_feature_names_out()`). Then create a small dataframe with two new (made-up) customers, one of which has missing values, and predict their churn probability with `model`.**

**QUESTION: Compare the cross-validated AUC with the test AUC and with the AUC of the leaky feature in section 2. Is the difference between CV and test worrying? What exactly does the pipeline protect you against?**

*Your answer:*

## 6 - Tuning the Pipeline with `GridSearchCV`

Because the whole pipeline is one estimator, we can tune the hyperparameters of the model **and** of the preprocessing steps at once. Parameters of a step are addressed with `<step name>__<parameter>`, e.g. `classifier__C` or `preprocessor__num__imputer__strategy`. The grid search itself also runs on the training set only.

**TASK: Run a `GridSearchCV` on `model` with `cv=cv` and `scoring="roc_auc"` over `classifier__C` in `[0.01, 0.1, 1, 10, 100]` and `preprocessor__num__imputer__strategy` in `["mean", "median"]`. Print the best parameters, the best CV score and the AUC of the best model on the test set.** *(Expected: best C = 100 with median imputation, best CV AUC ≈ 0.809 (all C ≥ 0.1 are within 0.001 of each other), test AUC ≈ 0.761.)*

**QUESTION: Does tuning help much? What does this tell you about the model? Why is `grid.best_score_` still not a fully unbiased estimate of the performance, and why do we need the test set?**

*Your answer:*

## 7 - Brainstorming: Spot the Leakage

For each scenario, decide whether there is data leakage, which type it is (target leakage, train-test contamination, ...), and how you would fix it.

### Scenario 1: Chest X-rays

A hospital builds a model that detects pneumonia on chest X-ray images. The dataset contains 10,000 images of 3,000 patients: many patients have several X-rays taken during the same hospital stay. The images are randomly split into 80% training and 20% test images. The model reaches 97% accuracy on the test set, but only 75% on images from a different hospital.

**QUESTION (Scenario 1): Is there data leakage? Which type? How would you split the data instead?**

*Your answer:*

### Scenario 2: Fraud detection with oversampling

Only 1% of the transactions in a credit card dataset are fraudulent. To deal with the class imbalance, a data scientist first **oversamples** the fraud cases (copying every fraud row 50 times) on the complete dataset, and then performs a 5-fold cross-validation. The recall of fraud cases is 99%.

**QUESTION (Scenario 2): What is wrong with this approach? Why is the recall so high? What is the correct way to deal with the imbalance?**

*Your answer:*

### Scenario 3: Forecasting daily ice-cream sales

A shop wants to forecast tomorrow's ice-cream sales. A student builds a dataset with one row per day for 3 years with the features *temperature of the day*, *day of the week* and *7-day moving average of the sales, centred on the day* (i.e. the average of the 3 days before, the day itself and the 3 days after). The model is evaluated with a shuffled 10-fold cross-validation and the error is very small.

**QUESTION (Scenario 3): Which features and which evaluation choice cause leakage? How would you build the features and the validation instead?** *(Hint: think about what is known at the moment you make the forecast.)*

*Your answer:*

## 8 - Summary

**QUESTION: Summarise what you learned in this exercise. Consider the target-leakage feature, the difference between unsupervised and supervised preprocessing leakage, and what pipelines do for you.**

*Your answer:*

## Great work!

----